# はじめに

このノートでは強化学習を理解するために、「ゼロから作るDeepLearning4(強化学習)」の書籍に従って、強化学習の基礎について理解を深める。ノートの中のコードは書籍のサポートページに記載されているコードを参考にさせて頂く。

- [ゼロから作るDeepLearning4(強化学習)](https://www.oreilly.co.jp/books/9784873119755/)
- [ゼロから作るDeepLearning4(強化学習)のコード](https://github.com/oreilly-japan/deep-learning-from-scratch-4/tree/master)

次の章に行く前に用語の整理をしておく。

- 価値: 報酬の期待値
- 行動価値: 行動に対して得られた報酬の期待値。QualityのQで表記
- 報酬: Rewardの$R$で表記
- 行動: Actionの$A$で表記
- 状態: Stateの$S$で表記
- 状態遷移確率: $p(s^{'}|s,a)$
- 報酬関数: $r(s,a,s^{'})$
- 方策: $\pi(a|s)$
- 収益: Returnではあるが$G$で表記される
- 状態価値関数: $\nu_{\pi}(s) = \mathbb{E}_{\pi} \left[ G_{t} | S_{t} = s \right] = \mathbb{E}_{\pi} \left[ \sum_{t=0}^{\infty} \gamma^{t} R_{t} | S_{t} = s \right]$


## 第3章: ベルマン方程式

マルコフ決定過程が確率的に振る舞う場合に、状態価値関数は簡単には計算できない。それを解決するのが状態価値関数についてのベルマン方程式である。ベルマン方程式はマルコフ決定過程において成り立つ強化学習アルゴリズムの基礎となる概念である。報酬の期待値の計算をおさらいしておく。定義は下記の通り。

- $\mathbb{E}[x] = \sum_x \sum_y p(x, y) r(x, y) = \sum_x \sum_y p(x) p(y | x) r(x, y)$ 

例の設定としては、サイコロとコインを利用する。サイコロの出目は$\frac{1}{6}$で、コインは$\frac{1}{2}$ではなく、サイコロの出目が偶数の場合は表が出やすいコインを利用し、奇数の場合は普通のコインを利用する。

- サイコロの出目が1(=1/6)のとき、コインの表(=1/2)
- サイコロの出目が1(=1/6)のとき、コインの裏(=1/2)
- サイコロの出目が2(=1/6)のとき、コインの表(=4/5)
- サイコロの出目が2(=1/6)のとき、コインの裏(=1/5)
- サイコロの出目が3(=1/6)のとき、コインの表(=1/2)
- サイコロの出目が3(=1/6)のとき、コインの裏(=1/2)
- サイコロの出目が4(=1/6)のとき、コインの表(=4/5)
- サイコロの出目が4(=1/6)のとき、コインの裏(=1/5)
- サイコロの出目が5(=1/6)のとき、コインの表(=1/2)
- サイコロの出目が5(=1/6)のとき、コインの裏(=1/2)
- サイコロの出目が6(=1/6)のとき、コインの表(=4/5)
- サイコロの出目が6(=1/6)のとき、コインの裏(=1/5)

$$
\begin{aligned}
\mathbb{E}[r(x, y)]
&= \sum_{x=1}^6 \sum_{y=0}^1 p(x, y) r(x, y) \\
&= 
 \left( \frac{1}{6} \cdot \frac{1}{2}  \cdot 1 \right) + 
 \left( \frac{1}{6} \cdot \frac{1}{2}  \cdot 0 \right) \\
 &+\left( \frac{1}{6} \cdot \frac{4}{5}  \cdot 2 \right) + 
 \left( \frac{1}{6} \cdot \frac{1}{5}  \cdot 0 \right)  \\
 &+\left( \frac{1}{6} \cdot \frac{1}{2}  \cdot 3 \right) + 
 \left( \frac{1}{6} \cdot \frac{1}{2}  \cdot 0 \right)  \\
 &+\left( \frac{1}{6} \cdot \frac{4}{5}  \cdot 4 \right) + 
 \left( \frac{1}{6} \cdot \frac{1}{5}  \cdot 0 \right)  \\
 &+\left( \frac{1}{6} \cdot \frac{1}{2}  \cdot 5 \right) + 
 \left( \frac{1}{6} \cdot \frac{1}{2}  \cdot 0 \right)   \\
 &+\left( \frac{1}{6} \cdot \frac{4}{5}  \cdot 6 \right) + 
 \left( \frac{1}{6} \cdot \frac{1}{5}  \cdot 0 \right) \\
&= 2.35
\end{aligned}
$$

In [5]:
import numpy as np

def get_coin_prob(dice):
    """サイコロの出目に応じてコインの確率分布を返す"""
    if dice % 2 == 0:
        # 偶数のとき
        return np.array([0.2, 0.8])
    else:
        # 奇数のとき
        return np.array([0.5, 0.5])

def expected_reward():
    x = np.arange(1, 7)  # サイコロの出目
    p_x = np.full(6, 1/6)  # サイコロの確率
    E = 0.0
    for i, dice in enumerate(x):
        print(i, dice)
        p_coin = get_coin_prob(dice)
        for coin in [0, 1]:
            reward = dice * coin
            E += p_x[i] * p_coin[coin] * reward
    return round(E, 2)

print(expected_reward())

0 1
1 2
2 3
3 4
4 5
5 6
2.35


収益は下記の通り定義される。連続タスクを想定すると、無限に続く報酬を仮定する。収益$G_{t}$は、時刻$t$以降に得られる報酬の総和である。割引率$\gamma$は0-1で定義される。

$$
\begin{aligned}
G_t = R_t + \gamma R_{t+1} + \gamma^2 R_{t+2} + \cdots
\tag{3.1}
\end{aligned}
$$

次に、収益$G_{t+1}$を考える。収益$G_{t+1}$は、時刻$t+1$以降に得られる報酬の総和である。

$$
\begin{aligned}
G_{t+1} = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} +  \cdots
\tag{3.2}
\end{aligned}
$$

3.1に3.2を代入すると

$$
\begin{aligned}
G_{t} &= R_{t} + \gamma(R_{t+1} + \gamma R_{t+2} + \cdots) \\
&= R_{t} + \gamma G_{t+1} 
\tag{3.3}
\end{aligned}
$$

のとおり、収益$G_{t},G_{t+1}$の関係性を導出できる。そして3.3を状態価値関数の定義に代入すると

$$
\begin{aligned}
\nu_{\pi}(s) 
&= \mathbb{E}_{\pi} \left[ G_{t} | S_{t}=s \right] \\
&= \mathbb{E}_{\pi} \left[ R_{t} + \gamma G_{t+1}  | S_{t}=s \right] \\
&= \mathbb{E}_{\pi} \left[ R_{t} | S_{t}=s \right] + \gamma \mathbb{E}_{\pi} \left[ G_{t+1} | S_{t}=s \right]\\
\tag{3.4}
\end{aligned}
$$



## 第1項
この3.4からさらに式変形を進めていくと、ベルマン方程式が導出できる。まず$\mathbb{E}_{\pi} \left[ R_{t} | S_{t}=s \right]$は何を意味しているのかを理解するところから始める。

今状態$s$であり、エージェントは方策$\pi(a|s)$に従って行動する。例えば3つの行動が取れるとして、エージェントは下記の確率分布に従って行動する。

$$
\begin{aligned}
\pi(a=a_{1}|s) = 0.2 \\
\pi(a=a_{2}|s) = 0.3 \\
\pi(a=a_{3}|s) = 0.5 \\
\end{aligned}
$$

そして、状態遷移確率に従って、新しい状態$s^{'}$に移動する。遷移先の候補が2つあるときは下記のようになる。

$$
\begin{aligned}
p(s^{'}=s_{1}|s, a=a_{1})=0.6 \\
p(s^{'}=s_{2}|s, a=a_{1})=0.4 \\
\end{aligned}
$$

最後に報酬が$r(s,a,s^{'})$という報酬関数で決まる。エージェントが0.2の確率で行動$a_{1}$を選び、0.6の確率で$s_{1}$に移動する。つまり、$\pi(a=a_{1}|s)p(s^{'}=s_{1}|s, a=a_{1})=0.2 \cdot 0.6 = 0.12$の確率で$r(s=s,a=a_{1},s^{'}=s_{1})$の報酬が得られる。期待値を計算するためには、他の選択肢についても同様に計算する必要があるため、期待値は下記のようになる。

$$
\begin{aligned}
\mathbb{E}_{\pi} \left[ R_{t} | S_{t}=s \right]
&= \sum_{a} \sum_{s'} \pi(a | s)  p(s' | s, a) r(s, a, s') \\
&= \sum_{a,s'} \pi(a | s)  p(s' | s, a) r(s, a, s')
\end{aligned}
$$

図解するとわかりよい。まず$\pi(a | s)$での選択があり、その選択に合わせてさらに$p(s' | s, a)$という選択がある。

```txt
s
 ├ a1: π(a=a1|s) = 0.2
 │ ├ p(s'=s1|s,a=a1)=0.6 -> r(s, a=a1, s'=s1)
 │ └ p(s'=s2|s,a=a1)=0.4 -> r(s, a=a1, s'=s2)
 │
 ├ a2: π(a=a2|s) = 0.3
 │ ├ p(s'=s1|s,a=a2)=?.? -> r(s, a=a2, s'=s1)
 │ └ p(s'=s2|s,a=a2)=?.? -> r(s, a=a2, s'=s2)
 │
 └ a3: π(a=a3|s) = 0.5
   ├ p(s'=s1|s,a=a3)=?.? -> r(s, a=a3, s'=s1)
   └ p(s'=s2|s,a=a3)=?.? -> r(s, a=a3, s'=s2)
```

つまり、すべての組み合わせに対して、「エージェントの行動の確率$\pi(a | s) $」「遷移先となる状態の確率$p(s' | s, a)$」「結果としての報酬関数$r(s, a, s')$」をかけ合わせ総和を取ったものをこの式は意味している。もう少し丁寧に導出してみる。

先ほどと同じく、時刻$t$で状態$s$ののケースを考える。1行目は条件付き確率の定義とは異なるため、違和感があるかもしれないが、すべての組み合わせに対して、計算をすることで期待値を算出することを前提に考えると、行動$a$があって、状態遷移$s→s'$するため、$s'$の数分インデックスを回すと考える。ただ、このままでは行動$a$の分が足りなくなるので、周辺化することで行動$a$を追加することで、行動$a$を計算に取り込む。3から4行目の式変形は、ノート末尾の補足を参照。あとは並び替えて、報酬関数を詳細化している。

$$
\begin{aligned}
\mathbb{E}[R_{t} | S_{t}=s] 
&= \sum_{s'} R_{t} p(S_{t+1} = s' | S_{t} = s)\\
&= \sum_{s'} R_{t} p(s'| s)\\
&= \sum_{s'} R_{t} \sum_{a} p(s', a | s)\\
&= \sum_{s'} R_{t} \sum_{a} \pi(a | s) p(s' | a, s)\\
&= \sum_{s'} \sum_{a} \pi(a | s) p(s' | a, s) r(s, a, s') \\
&= \sum_{a,s'} \pi(a | s) p(s' | a, s) r(s, a, s')
\end{aligned}
$$

## 第2項

次は第2項の$\gamma \mathbb{E}_{\pi} \left[ G_{t+1} | S_{t}=s \right]$を考える。$\gamma$は定数なので考える必要はなく、$\mathbb{E}_{\pi} \left[ G_{t+1} | S_{t}=s \right]$の部分を考える。

状態価値関数は、

$$
\begin{aligned}
\nu_{\pi}(s) = \mathbb{E}_{\pi} \left[ G_{t} | S_{t}=s \right] 
\end{aligned}
$$

であり、ここで考えたい第2項とは$G_{t}, G_{t+1}$という違いがある。ただ、この違いは時点が1つ異なるだけなので、$t = t+1$とすると、

$$
\begin{aligned}
\nu_{\pi}(s) = \mathbb{E}_{\pi} \left[ G_{t+1} | S_{t+1}=s \right]
\end{aligned}
$$

となり、状態$S_{t+1}$での状態価値関数といえる。ただ、考えたい第２項目は「時刻$t$でのひとつ先の$t+1$における収益の期待値」である。これは条件$S_{t}=s$を$S_{t+1}=s$の形に変えることで導出できる。たとえば、エージェントが$S_{t}=s$の状態にいて、エージェントが0.2の確率で$a_{1}$という行動を選択し、0.6の確率で$s_{1}$という状態に遷移するとする。

```txt
s
 ├ a1: π(a=a1|s) = 0.2
 │ ├ p(s'=s1|s,a=a1)=0.6 -> r(s, a=a1, s'=s1)
```
式で表すと$\pi(a=a_{1}|s)p(s^{'}=s_{1}|s, a=a_{1})=0.2 \cdot 0.6 = 0.12$の確率で$\nu_{\pi}(s_{1})=\mathbb{E}_{\pi} \left[ G_{t+1} | S_{t+1}=s_{1} \right]$に遷移する。このように1つ先の時点見ることで、次の状態の価値関数が得られる。$\mathbb{E}_{\pi} \left[ G_{t+1} | S_{t}=s \right]$という期待値を計算するときは、この計算をすべての組み合わせで計算して和を求めればよい。$\mathbb{E}[x | z] = \sum_y p(y | z) \mathbb{E}[x | y] $の関係を頭に入れておくとわかりよい。

$$
\begin{aligned}
\mathbb{E}[x | z] &= \sum_y p(y | z) \mathbb{E}[x | y]　\\ 
\mathbb{E}_{\pi}[G_{t+1} | S_t = s] &= \sum_{s'}  p(s' | s) \mathbb{E}_{\pi}[G_{t+1} | S_{t+1} = s'] \\
&= \sum_{s'} \sum_{a} p(s',a | s) \mathbb{E}_{\pi}[G_{t+1} | S_{t+1} = s'] \\
&= \sum_{s'} \sum_a \pi(a | s) p(s' | a, s) \mathbb{E}_{\pi}[G_{t+1} | S_{t+1} = s'] \\
&= \sum_{a} \pi(a | s) \sum_{s'} p(s' | a, s) \mathbb{E}_{\pi}[G_{t+1} | S_{t+1} = s'] \\
&= \sum_{a} \pi(a | s) \sum_{s'} p(s' | a, s) \nu_{\pi}(s') \\
&= \sum_{a,s'} \pi(a | s)  p(s' | a, s) \nu_{\pi}(s')
\end{aligned}
$$

これで$G_{t+1}$の期待値の具体的な計算式を得られた。

## ベルマン方程式

第1項目と第2項目を元の式に代入するとベルマン方程式が得られる。

$$
\begin{aligned}
\nu_{\pi}(s) 
&= \mathbb{E}_{\pi} \left[ G_{t} | S_{t}=s \right] \\
&= \mathbb{E}_{\pi} \left[ R_{t} + \gamma G_{t+1}  | S_{t}=s \right] \\
&= \mathbb{E}_{\pi} \left[ R_{t} | S_{t}=s \right] + \gamma \mathbb{E}_{\pi} \left[ G_{t+1} | S_{t}=s \right]\\
&= \sum_{a,s'} \pi(a | s) p(s' | a, s) r(s, a, s') + \gamma \sum_{a, s'} \pi(a | s) p(s' | a, s) \nu_{\pi}(s') \\
&= \sum_{a} \pi(a | s) \sum_{s'} p(s' | a, s) r(s, a, s') + \gamma \sum_{a} \pi(a | s) \sum_{s'} p(s' | a, s) \nu_{\pi}(s') \\
&= \sum_a \pi(a | s) \sum_{s'} p(s' | s, a) \Bigl\{ r(s, a, s') + \gamma v_{\pi}(s') \Bigr\}
\end{aligned}
$$

ベルマン方程式は「状態$s$の状態価値関数」と「その次の取るうる状態$s'$の状態価値関数」との関係を表した式であり、強化学習の問題を解くための重要な概念である。

## おまけ：条件付き確率の変換

$\mathbb{E}[x] = \sum_y \mathbb{E}[x | y] p(y)$が成り立つ。1行目は期待値の定義、2行目は$y$で周辺化を戻した同時確率、3行目は同時確率を乗法定理で条件付き確率にしたもの、4行目は条件付き期待値の定義を順番に適用すれば導出できる。

$$
\begin{aligned}
\mathbb{E}[x] 
&= \sum_{x}x p(x) \\ 
&= \sum_{x} x \sum_y p(x, y) \\ 
&= \sum_{x} \sum_y x p(x | y) p(y) \\ 
&= \sum_y \mathbb{E}[x | y] p(y) \\ 
\end{aligned}
$$


$\mathbb{E}[x | z] = \sum_y \mathbb{E}[x | y] p(y | z)$が成り立つ。その前に、条件付き確率の変換が出てくるので、そこを理解しておく。

数値の詳細は[「条件付き確率のおさらい」](https://sugiaki1989.github.io/statistical_note/note_CausalInference10/note_CausalInference10.html)を見るとして、この式変形$p(x,y|z) = p(x | y,z) p(y|z)$は下記の通り成り立つ。

- $P(Z=0)=0.04+0.16+0.06+0.18=0.44$
- $P(Y=0,Z=0)=0.04+0.06=0.10$
- $P(X=0,Y=0,Z=0) = 0.04$
- $P(X=0,Y=1,Z=0) = 0.16$
- $P(X=1,Y=0,Z=0) = 0.06$
- $P(X=1,Y=1,Z=0) = 0.18$
$$
\begin{aligned}
p(x,y|z)       &= p(x | y,z) p(y|z) \Leftrightarrow 0.09 = 0.40 \cdot 0.227 \\
p(x=0,y=0|z=0) &= \frac{p(x=0,y=0,z=0)}{p(z=0)} = \frac{0.04}{0.44} = 0.09 \\
p(x=0|y=0,z=0) &= \frac{p(x=0,y=0,z=0)}{p(y=0,z=0)} = \frac{0.04}{0.10} = 0.40 \\
p(y=0|z=0)     &= \frac{p(y=0,z=0)}{p(z=0)} = \frac{0.10}{0.44} = 0.227
\end{aligned}
$$

1行目は期待値の定義、2行目は$y$で条件付き確率の周辺化を戻した同時確率、3行目は条件付き確率を変換したもので、4行目は条件付き独立($x⊥z∣y$)の場合、$p(x∣y,z)=p(x∣y)$が成立、5行目は条件付き期待値の定義を順番に適用すれば導出できる。
$$
\begin{aligned}
\mathbb{E}[x | z]
&= \sum_{x} x p(x | z) \\
&= \sum_{x} x\sum_y p(x, y | z) \\
&= \sum_{x} \sum_y x p(x | y,z) p(y | z) \\
&= \sum_{x} \sum_y x p(x | y) p(y | z) \\
&= \sum_y \mathbb{E}[x | y] p(y | z)
\end{aligned}
$$


## ベルマン方程式の実例

第1項目と第2項目を元の式に代入するとベルマン方程式が得られる。

$$
\begin{aligned}
\nu_{\pi}(s) = \sum_a \pi(a | s) \sum_{s'} p(s' | s, a) \Bigl\{ r(s, a, s') + \gamma v_{\pi}(s') \Bigr\}
\end{aligned}
$$

ベルマン方程式は「状態$s$の状態価値関数」と「その次の取るうる状態$s'$の状態価値関数」との関係を表した式であり、強化学習の問題を解くための重要な概念である。

2025年7月15日
ちょっと別の内容を勉強する必要がでてきたので強化学習はSTOP
